# Chapter 3 - Lab 9: <font color='blue'>PydanticAI Agent</font>

**<font color='purple'>Goal</font>**:
In this lab, you will build a **financial analysis agent using PydanticAI** that compares the P/E (Price/Earnings) ratios of two companies — Apple (AAPL) and JPMorgan (JPM) — and produces a short investment memo.

PydanticAI takes a **type-safe, Pythonic** stance: agents are declared as module-level objects, tools are registered with a `@agent.tool` decorator, and the LLM provider is specified as a single string. Pydantic-style validation runs at every boundary.

In this lab you will see how concise a tool-using agent can be when the framework treats Pydantic types as first-class citizens.

This is the same reference task used across every framework lab in Chapter 3 — comparing all of them on the *same* task makes the differences in API style, abstractions, and ergonomics easy to spot.

**<font color='purple'>Tech stack</font>**:

* **PydanticAI** (`pydantic-ai`) — `Agent`, `RunContext`, `@agent.tool`.
* **OpenAI** `gpt-4o-mini` (via the `'openai:gpt-4o-mini'` provider string).
* **Pydantic** — used both for tool I/O and for output validation.

You will need an OpenAI API key with some credits available.

## 1. Install packages

Install the framework and its dependencies.

In [1]:
%pip install -q pydantic-ai pydantic python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.9/103.9 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.7/859.7 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.8/118.8 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.8/474.8 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/

## 2. Set up the API key(s)

This lab needs the following key(s):

  * **`OPENAI_API_KEY`** — your OpenAI key

If you are running this notebook in **Google Colab**, add each key in the left vertical menu under the *key* icon, using the names above.

If you are running locally, set the same names as environment variables (or load them from a `.env` file).

In [2]:
import os

try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally — assume env vars are already set (e.g. via .env).
    pass

## 3. Bootstrap the shared task setup

Every framework lab in this chapter shares the same task, tools, finance dataset, and prompts. These are factored out into `common.py`. If you have cloned the book's repository, `common.py` is already on disk; otherwise the cell below downloads it for you.

In [3]:
import os, urllib.request

if not os.path.exists('common.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/PacktPublishing/Building-AI-Agents-for-Finance-/main/Chapter%203/common.py',
        'common.py',
    )

from common import (
    get_stock_data,
    compute_pe,
    finance_data,
    system_message,
    input_message,
)

print('Tools loaded. Reference task:')
print(' ', input_message)

Tools loaded. Reference task:
  Compare Apple (AAPL) and JPMorgan (JPM) on P/E ratios and summarize in a memo.


## 4. Declare the agent

PydanticAI agents are declared at module scope. The first argument names the model provider; the `system_prompt` is just a string. Notice how short this is compared to the AutoGen or ADK setup.

In [4]:
from pydantic_ai import Agent, RunContext

agent = Agent('openai:gpt-4o-mini', system_prompt=system_message)

## 5. Register tools with `@agent.tool`

Tools are registered against a specific agent instance. The first parameter is always a `RunContext` (which carries deps, retries, model info), even if your tool does not need it. Return types may be Pydantic models or plain JSON-serialisable values.

In [5]:
@agent.tool
def get_stock_data_tool(ctx: RunContext, ticker: str) -> dict:
    """Get stock data for a ticker."""
    result = get_stock_data(ticker)
    return {'ticker': ticker.upper(), 'price': result.price, 'eps': result.eps}


@agent.tool
def compute_pe_ratio_tool(ctx: RunContext, price: float, eps: float) -> float:
    """Compute the P/E ratio."""
    return compute_pe(price, eps)

## 6. Run the agent

`agent.run_sync` blocks until completion and returns a result whose `.output` holds the model's final answer. For async use `agent.run`.

In [8]:
result = await agent.run(input_message)
print(result.output)

### Memo

**To:** [Recipient]  
**From:** [Your Name]  
**Date:** [Today's Date]  
**Subject:** Comparison of P/E Ratios for Apple (AAPL) and JPMorgan (JPM)

**Overview:**
This memo provides a comparative analysis of the Price-to-Earnings (P/E) ratios for Apple Inc. (AAPL) and JPMorgan Chase & Co. (JPM).

**P/E Ratio Calculation:**
The P/E ratio is calculated using the formula:
\[ 
\text{P/E Ratio} = \frac{\text{Price per Share}}{\text{Earnings per Share (EPS)}}
\]

**Results:**
- **Apple (AAPL):**
  - Price: $195.30
  - EPS: $6.67
  - **P/E Ratio: 29.28**

- **JPMorgan (JPM):**
  - Price: $148.70
  - EPS: $12.61
  - **P/E Ratio: 11.79**

**Analysis:**
- Apple's P/E ratio of **29.28** suggests that investors are willing to pay a premium for its earnings, indicative of a high growth expectation or market sentiment favoring technology stocks.
- Conversely, JPMorgan's P/E ratio of **11.79** implies a more conservative valuation, likely reflecting the more stable and lower growth nature of

## 7. Results

You should see the agent call `get_stock_data_tool` (once per ticker), then `compute_pe_ratio_tool` (once per ticker), and finally produce a short memo comparing the two.

**What to notice about PydanticAI specifically:**

* The **most Pythonic** of the frameworks in this chapter — the agent is a regular module-level object you can configure, share, and test like any other.
* Pydantic types appear at every boundary (system prompt schema, tool I/O, validated outputs), which is excellent for production reliability.
* The `'provider:model'` string convention (also used by LangChain) means provider swaps are one-line changes.
* Trade-off: smaller community than LangChain or LlamaIndex; fewer pre-built integrations. Pick PydanticAI when type safety and explicitness matter more than ecosystem breadth.